# 01 — Project Validation Notebook

**Crypto Market Research Laboratory — Prompt 01**

---

## Purpose

This notebook validates that the **Prompt 01 infrastructure** is correctly installed and functional.

It does NOT perform any trading research. It does NOT use real or fake market data.

It validates:

1. Python environment and dependency versions
2. Project configuration loading and validation
3. Research run creation and metadata generation
4. Directory structure integrity
5. Logging initialization
6. Core domain object integrity
7. Git metadata capture

---

## What success means

If all cells execute without error and the final cell displays `✅ ALL CHECKS PASSED`,
Prompt 01 is complete and the project is ready for **Prompt 02 — Binance Data Ingestion**.

## What failure means

Any error in any cell **must not be hidden**. The notebook will fail visibly.
Fix the error at its source — do not catch exceptions to display a fake success.

## Step 1: Environment Check

Verify the Python version and that all required packages are installed.

In [ ]:
import sys
import platform

print(f"Python version : {sys.version}")
print(f"Platform       : {platform.system()} {platform.release()}")
print(f"Machine        : {platform.machine()}")

# Verify minimum Python version
assert sys.version_info >= (3, 11), (
    f"Python 3.11+ required. Got: {sys.version_info.major}.{sys.version_info.minor}"
)
print("\n✅ Python version OK")

In [ ]:
# Verify critical packages are importable and record versions
from importlib.metadata import version, PackageNotFoundError

required_packages = ["pydantic", "pyyaml", "structlog", "rich"]
missing = []

for pkg in required_packages:
    try:
        v = version(pkg)
        print(f"  {pkg:20s} {v}")
    except PackageNotFoundError:
        missing.append(pkg)
        print(f"  {pkg:20s} *** MISSING ***")

if missing:
    raise RuntimeError(
        f"Required packages not installed: {missing}\n"
        f"Run: pip install -e '.[dev]'"
    )

print("\n✅ All required packages installed")

## Step 2: Project Package Import

Verify that the `crypto_research` package is importable from the editable install.

In [ ]:
import crypto_research

print(f"crypto_research package version: {crypto_research.__version__}")

# Import core components
from crypto_research.config import load_config, load_default_config, find_project_root
from crypto_research.core import (
    Candle, Signal, Order, Fill, Position, Trade,
    Timeframe, SignalDirection, OrderSide, OrderType,
    RiskDecision, StrategyMetadata, ResearchRun,
    ConfigurationError, DataIntegrityError, LookAheadBiasError,
)
from crypto_research.research.run_manager import RunManager
from crypto_research.utils.logging import setup_logging
from crypto_research.utils.environment import get_full_environment_info

# Verify interfaces are importable
from crypto_research.data.interface import DataProvider
from crypto_research.strategies.interface import Strategy
from crypto_research.execution.interface import ExecutionEngine
from crypto_research.risk.interface import RiskManager

print("✅ All imports successful")

## Step 3: Find Project Root and Configuration

Locate the project root and verify the configuration file exists.

In [ ]:
from pathlib import Path

project_root = find_project_root()
config_path = project_root / "config" / "config.yaml"

print(f"Project root   : {project_root}")
print(f"Config file    : {config_path}")
print(f"Config exists  : {config_path.exists()}")

if not config_path.exists():
    raise FileNotFoundError(
        f"Configuration file not found at: {config_path}\n"
        f"This is a required file. Do not proceed without it."
    )

print("\n✅ Project root and config file found")

## Step 4: Load and Validate Configuration

Load the YAML configuration and validate it against the Pydantic schema.

**If this step fails**, there is a problem with `config/config.yaml`.
Fix the configuration file — do not modify Python code to bypass validation.

In [ ]:
config = load_config(config_path)

print("Configuration Summary")
print("=" * 50)
print(f"  Project name    : {config.project.name}")
print(f"  Project version : {config.project.version}")
print(f"  Market          : {config.research.market}")
print(f"  Data source     : {config.research.data_source}")
print(f"  Assets          : {config.assets}")
print(f"  Timeframes      : {config.timeframes}")
print()
print("Risk Configuration")
print(f"  Risk/Reward     : 1 : {config.risk.risk_reward_ratio}")
print(f"  Risk per trade  : {config.risk.risk_per_trade_pct}%")
print(f"  Max positions   : {config.risk.max_concurrent_positions}")
print(f"  Daily loss limit: {config.risk.max_daily_loss_pct}%")
print()
print("Execution Configuration")
print(f"  Fee rate        : {config.execution.fee_rate} (null = not yet configured)")
print(f"  Slippage model  : {config.execution.slippage_model} (null = not yet configured)")
print()
print("✅ Configuration loaded and validated successfully")

## Step 5: Create a Research Run

Create a new research run, which generates:
- A unique run ID
- A run directory under `results/`
- `config_snapshot.yaml` — exact config used
- `metadata.json` — full environment and run metadata

In [ ]:
manager = RunManager()
run = manager.create_run(config, config_path=config_path)

print(f"Run ID          : {run.run_id}")
print(f"Created at      : {run.created_at.isoformat()}")
print(f"Run directory   : {run.run_directory}")
print(f"Git commit      : {run.git_commit}")
print(f"Python version  : {run.python_version.split()[0]}")
print()
print("Package versions recorded:")
for pkg, ver in run.package_versions.items():
    print(f"  {pkg:20s} {ver}")

print("\n✅ Research run created successfully")

## Step 6: Initialize Logging

Initialize structured logging for this validation run.

In [ ]:
log_dir = Path(run.run_directory) / "logs"

logger = setup_logging(
    run_id=run.run_id,
    log_dir=log_dir,
    level=config.logging.level,
    format=config.logging.format,
)

logger.info("Notebook validation started", run_id=run.run_id, step="logging_init")

log_file = log_dir / "research.log"
assert log_file.exists(), f"Log file not created at: {log_file}"

print(f"Log file created: {log_file}")
print("\n✅ Logging initialized")

## Step 7: Validate Directory Structure

Verify that the expected project and run directory structure is complete.

In [ ]:
checks = []

def check(description: str, condition: bool, path: str = ""):
    """Record a check result."""
    status = "✅" if condition else "❌"
    label = f"{status} {description}"
    if path:
        label += f"  →  {path}"
    print(label)
    checks.append((description, condition))

# Project root directories
check("src/ exists", (project_root / "src").is_dir(), str(project_root / "src"))
check("config/ exists", (project_root / "config").is_dir())
check("notebooks/ exists", (project_root / "notebooks").is_dir())
check("tests/ exists", (project_root / "tests").is_dir())
check("data/ exists", (project_root / "data").is_dir())
check("results/ exists", (project_root / "results").is_dir())
check("logs/ exists", (project_root / "logs").is_dir())
check("docs/ exists", (project_root / "docs").is_dir())
check("scripts/ exists", (project_root / "scripts").is_dir())
check("pyproject.toml exists", (project_root / "pyproject.toml").is_file())
check("config.yaml exists", config_path.is_file())

# Source package structure
pkg_root = project_root / "src" / "crypto_research"
for subpkg in ["config", "core", "data", "strategies", "execution", "risk", "research", "reporting", "utils"]:
    check(f"src/crypto_research/{subpkg}/ exists", (pkg_root / subpkg).is_dir())

# Run directory structure
run_dir = Path(run.run_directory)
for subdir in ["logs", "trades", "charts", "reports", "summary"]:
    check(f"run/{subdir}/ exists", (run_dir / subdir).is_dir())
check("run/config_snapshot.yaml exists", (run_dir / "config_snapshot.yaml").is_file())
check("run/metadata.json exists", (run_dir / "metadata.json").is_file())

# Summary
failed = [d for d, ok in checks if not ok]
if failed:
    raise AssertionError(
        f"Directory structure check FAILED. Missing:\n" + "\n".join(f"  - {f}" for f in failed)
    )
print(f"\n✅ All {len(checks)} directory checks passed")

## Step 8: Validate Core Domain Objects

Verify that domain objects instantiate correctly and enforce their invariants.

In [ ]:
from datetime import datetime, timezone
import traceback

domain_checks = []

def domain_check(name: str, fn):
    try:
        fn()
        print(f"✅ {name}")
        domain_checks.append((name, True))
    except Exception as e:
        print(f"❌ {name}: {e}")
        domain_checks.append((name, False))

NOW = datetime(2026, 9, 15, 0, 0, 0, tzinfo=timezone.utc)

# Candle: valid
domain_check("Candle: valid OHLCV instantiation", lambda: Candle(
    timestamp=NOW, asset="BTCUSDT", timeframe=Timeframe.M1,
    open=50000, high=50500, low=49500, close=50200, volume=100
))

# Candle: invalid OHLC
def candle_invalid_ohlc():
    try:
        Candle(timestamp=NOW, asset="BTCUSDT", timeframe=Timeframe.M1,
               open=50000, high=49000, low=49500, close=50200, volume=100)
        raise AssertionError("Should have raised ValueError")
    except ValueError:
        pass  # expected
domain_check("Candle: OHLC violation raises ValueError", candle_invalid_ohlc)

# Signal: valid
domain_check("Signal: valid instantiation", lambda: Signal(
    timestamp=NOW, strategy_name="test", asset="BTCUSDT",
    timeframe=Timeframe.M5, direction=SignalDirection.LONG, strength=0.8
))

# Timeframe: valid parsing
domain_check("Timeframe: parse '1h'", lambda: Timeframe.from_string("1h"))

# Timeframe: invalid parsing
def timeframe_invalid():
    try:
        Timeframe.from_string("99x")
        raise AssertionError("Should have raised ValueError")
    except ValueError:
        pass
domain_check("Timeframe: invalid value raises ValueError", timeframe_invalid)

# RiskDecision: rejected without size
domain_check("RiskDecision: rejection with reason",
    lambda: RiskDecision(approved=False, reason="Daily loss limit reached"))

# Domain failures?
domain_failures = [n for n, ok in domain_checks if not ok]
if domain_failures:
    raise AssertionError(f"Domain object checks failed:\n" + "\n".join(f"  - {f}" for f in domain_failures))

print(f"\n✅ All {len(domain_checks)} domain object checks passed")

## Step 9: Validate Fail-Fast Behavior

Confirm that the system raises clear errors on invalid input — no silent fallbacks.

In [ ]:
import tempfile

failfast_checks = []

def failfast_check(name: str, fn):
    try:
        fn()
        print(f"✅ {name}")
        failfast_checks.append((name, True))
    except Exception as e:
        print(f"❌ {name}: {e}")
        failfast_checks.append((name, False))

# Missing config file
def test_missing_config():
    try:
        load_config("/tmp/nonexistent_config_abc123.yaml")
        raise AssertionError("Should have raised ConfigurationError")
    except ConfigurationError:
        pass
failfast_check("Missing config raises ConfigurationError", test_missing_config)

# Empty config file
def test_empty_config():
    with tempfile.NamedTemporaryFile(suffix=".yaml", mode="w") as f:
        f.write("")
        f.flush()
        try:
            load_config(f.name)
            raise AssertionError("Should have raised ConfigurationError")
        except ConfigurationError:
            pass
failfast_check("Empty config raises ConfigurationError", test_empty_config)

# No fake market data
def test_no_fake_trades():
    trades_dir = run_dir / "trades"
    files = list(trades_dir.iterdir())
    assert files == [], f"Found fake trade files: {files}"
failfast_check("No fake trade files in run directory", test_no_fake_trades)

# Git metadata integrity
import re
def test_git_integrity():
    valid_hash = re.compile(r"^[0-9a-f]{40}$")
    gc = run.git_commit
    assert gc == "unavailable" or valid_hash.match(gc), (
        f"Fabricated git commit: {gc}"
    )
failfast_check("Git commit is real hash or 'unavailable'", test_git_integrity)

# Summary
failfast_failures = [n for n, ok in failfast_checks if not ok]
if failfast_failures:
    raise AssertionError(f"Fail-fast checks failed:\n" + "\n".join(f"  - {f}" for f in failfast_failures))

print(f"\n✅ All {len(failfast_checks)} fail-fast behavior checks passed")

## Step 10: Display Generated Artifacts

Show what was created by this validation run.

In [ ]:
import json

print("Generated Artifacts")
print("=" * 60)
print(f"Run ID           : {run.run_id}")
print(f"Run directory    : {run.run_directory}")
print()

# Show metadata.json content
metadata_path = Path(run.run_directory) / "metadata.json"
with open(metadata_path) as f:
    metadata = json.load(f)

print("metadata.json contents:")
# Show key fields only (not full config snapshot)
for key in ["run_id", "created_at", "git_commit", "python_version", "assets", "timeframes"]:
    print(f"  {key:20s}: {metadata[key]}")

print()
print("Files created in run directory:")
for f in sorted(Path(run.run_directory).rglob("*")):
    if f.is_file():
        rel = f.relative_to(run.run_directory)
        print(f"  {rel}  ({f.stat().st_size} bytes)")

## Final Result

In [ ]:
all_check_lists = checks + domain_checks + failfast_checks
total = len(all_check_lists)
passed = sum(1 for _, ok in all_check_lists if ok)

print()
print("=" * 60)
if passed == total:
    print(f"✅ ALL CHECKS PASSED ({passed}/{total})")
    print()
    print("Prompt 01 is COMPLETE.")
    print(f"Research run ID: {run.run_id}")
    print()
    print("The project is ready for:")
    print("  PROMPT 02 — REAL BINANCE DATA INGESTION AND DATA QUALITY")
else:
    failed_checks = [d for d, ok in all_check_lists if not ok]
    raise AssertionError(
        f"VALIDATION FAILED: {passed}/{total} checks passed.\n"
        f"Failed checks:\n" + "\n".join(f"  - {f}" for f in failed_checks)
    )
print("=" * 60)